In [2]:
%reset -f
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
import os
warnings.filterwarnings('ignore')
os.chdir(r'G:\Kuangyu_Temp\Outsource\description')
# 设置中文字体支持
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 设置绘图风格
sns.set_style("whitegrid")
sns.set_palette("husl")

print("=" * 80)
print("企业外包行为描述性分析")
print("=" * 80)

企业外包行为描述性分析


In [3]:


# ============================================================================
# Step 1: 数据读取和清洗
# ============================================================================
print("\n[Step 1] 读取和清洗数据...")

# 读取数据
df = pd.read_stata(r'G:\Kuangyu_Temp\Outsource\lenth9.dta')
print(f"原始数据规模: {len(df):,} 行")



# 基本统计
print(f"\n企业数量: {df['firm_id'].nunique():,}")
print(f"产品数量: {df['product_id'].nunique():,}")
print(f"总交易额: {df['v'].sum()/1e9:.2f} 十亿元")

# ============================================================================
# Step 2: 识别外包产品
# ============================================================================
print("\n[Step 2] 识别外包产品...")

# 对每个企业-产品-日期组合，判断是否同时有买入和卖出
firm_product_summary = df.groupby(['firm_id', 'product_id', 'year']).agg({
    'is_output': lambda x: (x == 1).any() and (x == 0).any(),  # 既买又卖
    'v': 'sum'
}).reset_index()

firm_product_summary.columns = ['firm_id', 'product_id', 'year', 'is_outsourcing', 'total_value']

# 合并回原数据
df = df.merge(
    firm_product_summary[['firm_id', 'product_id', 'year', 'is_outsourcing']], 
    on=['firm_id', 'product_id', 'year'],
    how='left'
)

print(f"外包产品-企业-年份组合数: {df[df['is_outsourcing']].groupby(['firm_id', 'product_id', 'year']).ngroups:,}")

# ============================================================================
# Step 3: 构建企业层面汇总表
# ============================================================================
print("\n[Step 3] 构建企业层面汇总表...")

# 分别计算产出和投入
output_df = df[df['is_output'] == 1].copy()
input_df = df[df['is_output'] == 0].copy()

# 3.1 总产出和总投入
firm_totals = df.groupby(['firm_id', 'year']).agg({
    'v': lambda x: x[df.loc[x.index, 'is_output'] == 1].sum() if len(x[df.loc[x.index, 'is_output'] == 1]) > 0 else 0,
}).reset_index()
firm_totals.columns = ['firm_id', 'year', 'total_output']

firm_totals['total_input'] = df.groupby(['firm_id', 'year']).apply(
    lambda x: x.loc[x['is_output'] == 0, 'v'].sum()
).values

# 3.2 外包产品的产出和投入
outsourcing_output = input_df[input_df['is_outsourcing']].groupby(['firm_id', 'year'])['v'].sum().reset_index()
outsourcing_output.columns = ['firm_id', 'year', 'outsourcing_output']

outsourcing_input = input_df[input_df['is_outsourcing']].groupby(['firm_id', 'year'])['v'].sum().reset_index()
outsourcing_input.columns = ['firm_id', 'year', 'outsourcing_input']

# 3.3 产品数量统计
product_counts = output_df.groupby(['firm_id', 'year'])['product_id'].nunique().reset_index()
product_counts.columns = ['firm_id', 'year', 'n_products']

outsourcing_product_counts = output_df[output_df['is_outsourcing']].groupby(['firm_id', 'year'])['product_id'].nunique().reset_index()
outsourcing_product_counts.columns = ['firm_id', 'year', 'n_outsourcing_products']

# 3.4 主要产出产品（销售额最大）
main_product = output_df.groupby(['firm_id', 'year', 'product_id'])['v'].sum().reset_index()
main_product = main_product.sort_values(['firm_id', 'year', 'v'], ascending=[True, True, False])
main_product = main_product.groupby(['firm_id', 'year']).first().reset_index()
main_product = main_product[['firm_id', 'year', 'product_id', 'v']]
main_product.columns = ['firm_id', 'year', 'main_product', 'main_product_sales']

# 3.5 主要外包产品（外包销售额最大）
main_outsourcing = input_df[input_df['is_outsourcing']].groupby(['firm_id', 'year', 'product_id'])['v'].sum().reset_index()
main_outsourcing = main_outsourcing.sort_values(['firm_id', 'year', 'v'], ascending=[True, True, False])
main_outsourcing = main_outsourcing.groupby(['firm_id', 'year']).first().reset_index()
main_outsourcing = main_outsourcing[['firm_id', 'year', 'product_id', 'v']]
main_outsourcing.columns = ['firm_id', 'year', 'main_outsourcing_product', 'main_outsourcing_sales']

# 合并所有信息
firm_agg = firm_totals.copy()
firm_agg = firm_agg.merge(outsourcing_output, on=['firm_id', 'year'], how='left')
firm_agg = firm_agg.merge(outsourcing_input, on=['firm_id', 'year'], how='left')
firm_agg = firm_agg.merge(product_counts, on=['firm_id', 'year'], how='left')
firm_agg = firm_agg.merge(outsourcing_product_counts, on=['firm_id', 'year'], how='left')
firm_agg = firm_agg.merge(main_product, on=['firm_id', 'year'], how='left')
firm_agg = firm_agg.merge(main_outsourcing, on=['firm_id', 'year'], how='left')

# 填充缺失值
firm_agg['outsourcing_output'] = firm_agg['outsourcing_output'].fillna(0)
firm_agg['outsourcing_input'] = firm_agg['outsourcing_input'].fillna(0)
firm_agg['n_outsourcing_products'] = firm_agg['n_outsourcing_products'].fillna(0)

# 计算外包强度
firm_agg['outsourcing_intensity'] = firm_agg['outsourcing_output'] / firm_agg['total_output']
firm_agg['outsourcing_intensity'] = firm_agg['outsourcing_intensity'].fillna(0)

# 3.6 识别中间商（外包产品占产出>90%）
firm_agg['is_intermediary'] = (firm_agg['outsourcing_intensity'] > 0.90).astype(int)

print(f"\n企业-年份观测数: {len(firm_agg):,}")
print(f"有外包行为的企业-年份观测数: {(firm_agg['outsourcing_intensity'] > 0).sum():,}")
print(f"中间商企业-年份观测数: {firm_agg['is_intermediary'].sum():,}")
print(f"中间商占比: {firm_agg['is_intermediary'].mean()*100:.2f}%")

# 保存汇总表
firm_agg.to_stata('firm_aggregate_table.dta', index=False)
print("\n企业汇总表已保存至: firm_aggregate_table.dta")

# 显示样本
print("\n企业汇总表样本:")
print(firm_agg.head(10).to_string())

print("\n" + "=" * 80)
print("Step 3 完成！")
print("=" * 80)



[Step 1] 读取和清洗数据...
原始数据规模: 465,487,031 行

企业数量: 7,191,877
产品数量: 2,778
总交易额: 825547.64 十亿元

[Step 2] 识别外包产品...
外包产品-企业-年份组合数: 35,320,559

[Step 3] 构建企业层面汇总表...

企业-年份观测数: 13,094,906
有外包行为的企业-年份观测数: 7,716,925
中间商企业-年份观测数: 1,631,747
中间商占比: 12.46%


TypeError: DataFrame.to_stata() got an unexpected keyword argument 'index'

In [4]:
firm_agg.to_stata('firm_aggregate_table.dta', write_index=False)